# EDA — Mega Sena Analytics
**Análise Exploratória dos sorteios históricos da Mega Sena (1996–2026)**

Dados: `data/sorteios.json` — coletados via API da Caixa Econômica Federal.
Plots: **Plotly** — interativos: hover para detalhes, zoom, pan e export PNG pelo menu da figura.

---

### Sobre este notebook

A Mega Sena é um sorteio probabilístico. Isso significa que cada concurso é **independente** dos anteriores — o histórico não determina o próximo resultado.

Este notebook não tenta prever números. Seu objetivo é **descrever o que já aconteceu** e testar empiricamente crenças comuns sobre o jogo. Ao final, você saberá quais padrões existem de fato nos dados e quais são simplesmente produtos do acaso.

As análises estão organizadas em quatro blocos:

| Bloco | Seções | Pergunta central |
|---|---|---|
| Distribuição das dezenas | 1–4 | As dezenas se distribuem uniformemente? |
| Comportamento temporal | 5–6 | A frequência das dezenas muda com o tempo? |
| Co-ocorrência | 7 | Alguns pares aparecem juntos mais do que o esperado? |
| Contexto externo e correlações | 8–10 | Local, ordem e acúmulo influenciam os resultados? |

In [44]:
import json
import warnings
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)

warnings.filterwarnings("ignore")

# -- Paths
ROOT      = Path().resolve().parents[1]
DATA_PATH = ROOT / "data" / "sorteios.json"
EXPORTS   = ROOT / "python" / "exports"
EXPORTS.mkdir(exist_ok=True)

# -- Paleta global
PRIMARY   = "#1a1a2e"
ACCENT    = "#e94560"
HIGHLIGHT = "#f5a623"
MUTED     = "#8892b0"
BG        = "#f8f9fa"

# -- Template Plotly customizado
LAYOUT = dict(
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    font=dict(family="DejaVu Sans, sans-serif", color=PRIMARY),
    title_font=dict(size=16, color=PRIMARY),
    legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0),
    hoverlabel=dict(bgcolor="white", font_size=12),
    margin=dict(t=70, b=50, l=60, r=30),
)

DEZENA_COLS  = ["d1", "d2", "d3", "d4", "d5", "d6"]
SORTEIO_COLS = ["s1", "s2", "s3", "s4", "s5", "s6"]

from scipy.stats import spearmanr, chi2_contingency, ks_2samp, ttest_1samp

print("Imports OK")

Imports OK


## Dados

**Fonte:** API pública da Caixa Econômica Federal
**Cobertura:** concurso 1 (11/03/1996) ao mais recente

Cada linha representa um sorteio. Campos principais:

| Campo | Descrição |
|---|---|
| `d1`–`d6` | Dezenas sorteadas em **ordem crescente** (como no volante) |
| `s1`–`s6` | Dezenas na **ordem de saída ao vivo** |
| `local` / `cidade` | Local físico de realização do sorteio |
| `acumulado` | `True` se o prêmio principal não foi ganho e acumulou |
| `premio_total_sena` | Valor total da faixa sena (R$) |
| `ganhadores_6/5/4` | Número de ganhadores por faixa de acerto |

O DataFrame resultante tem uma linha por concurso e é ordenado pelo número do concurso.

In [ ]:
raw  = json.load(open(DATA_PATH))
rows = []

for c in raw:
    rateo   = {r["faixa"]: r for r in c.get("listaRateioPremio", [])}
    dezenas = sorted([int(x) for x in c["listaDezenas"]])
    sorteio = [int(x) for x in c["dezenasSorteadasOrdemSorteio"]]
    rows.append({
        "numero":       c["numero"],
        "data":         pd.to_datetime(c["dataApuracao"], dayfirst=True),
        "acumulado":    bool(c["acumulado"]),
        "especial":     c["indicadorConcursoEspecial"] != 1,
        "local":        c.get("localSorteio", ""),
        "cidade":       c.get("nomeMunicipioUFSorteio", ""),
        "d1": dezenas[0], "d2": dezenas[1], "d3": dezenas[2],
        "d4": dezenas[3], "d5": dezenas[4], "d6": dezenas[5],
        "s1": sorteio[0], "s2": sorteio[1], "s3": sorteio[2],
        "s4": sorteio[3], "s5": sorteio[4], "s6": sorteio[5],
        "valor_arrecadado":        c.get("valorArrecadado", 0.0),
        "valor_acumulado_proximo": c.get("valorAcumuladoProximoConcurso", 0.0),
        "valor_estimado_proximo":  c.get("valorEstimadoProximoConcurso", 0.0),
        "premio_total_sena": rateo.get(1, {}).get("valorPremio", 0.0)
        "ganhadores_6": rateo.get(1, {}).get("numeroDeGanhadores", 0),
        "premio_6":     rateo.get(1, {}).get("valorPremio", 0.0),
        "ganhadores_5": rateo.get(2, {}).get("numeroDeGanhadores", 0),
        "premio_5":     rateo.get(2, {}).get("valorPremio", 0.0),
        "ganhadores_4": rateo.get(3, {}).get("numeroDeGanhadores", 0),
        "premio_4":     rateo.get(3, {}).get("valorPremio", 0.0),
    })

df = pd.DataFrame(rows).sort_values("numero").reset_index(drop=True)

def all_dezenas(d=df):
    return d[DEZENA_COLS].values.flatten()

print(f"{len(df)} concursos | {df['data'].min().date()} -> {df['data'].max().date()}")
df.head(3)

3002 concursos | 1996-03-11 -> 2026-05-16


,numero,data,acumulado,especial,local,cidade,d1,d2,d3,d4,d5,d6,s1,s2,s3,s4,s5,s6,valor_arrecadado,valor_acumulado_proximo,valor_estimado_proximo,premio_total_sena,ganhadores_6,premio_6,ganhadores_5,premio_5,ganhadores_4,premio_4
0,1,1996-03-11,True,False,Auditório,"Brasília, DF",4,5,30,33,41,52,41,5,4,52,30,33,0.0,1714650.23,0.0,0.0,0,0.00,17,39158.92,2016,330.21
1,2,1996-03-18,False,False,Caminhão da Sorte,"Belo Horizonte, MG",9,37,39,41,43,49,9,39,37,49,43,41,0.0,0.00,0.0,0.0,1,2307162.23,65,14424.02,4488,208.91
2,3,1996-03-25,False,False,Auditório,"Brasília, DF",10,11,29,30,36,47,36,30,10,11,29,47,0.0,0.00,0.0,0.0,2,391192.51,62,10515.93,4261,153.01


In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3002 entries, 0 to 3001
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   numero                   3002 non-null   int64         
 1   data                     3002 non-null   datetime64[us]
 2   acumulado                3002 non-null   bool          
 3   especial                 3002 non-null   bool          
 4   local                    3002 non-null   str           
 5   cidade                   3002 non-null   str           
 6   d1                       3002 non-null   int64         
 7   d2                       3002 non-null   int64         
 8   d3                       3002 non-null   int64         
 9   d4                       3002 non-null   int64         
 10  d5                       3002 non-null   int64         
 11  d6                       3002 non-null   int64         
 12  s1                       3002 non-null   int6

---

## Bloco 1 — Distribuição das dezenas

> **Pergunta central:** as dezenas de 1 a 60 são sorteadas com a mesma frequência?

Se o sorteio for aleatório e uniforme, cada dezena deveria aparecer em aproximadamente **10% dos concursos** (6 tiradas de 60, sem reposição). Com mais de 3.000 sorteios, temos volume estatístico suficiente para verificar isso empiricamente.

As seções 1 a 4 analisam a distribuição sob quatro ângulos complementares: **frequência individual**, **agrupamento em faixas**, **composição par/ímpar** e **soma agregada**.

### 1. Frequência histórica de cada dezena

Quantas vezes cada número de 1 a 60 foi sorteado no total do histórico. A linha tracejada marca a média — o ponto onde todas as dezenas estariam se a distribuição fosse perfeitamente uniforme.

> Flutuações em torno da média são **esperadas** em qualquer processo aleatório finito. A questão relevante é: o desvio observado é compatível com variância aleatória, ou alguma dezena desvia tanto que sugere um viés real no mecanismo de sorteio?

In [47]:
freq   = Counter(all_dezenas())
nums   = list(range(1, 61))
counts = [freq[n] for n in nums]
media  = np.mean(counts)

top5 = sorted(range(1, 61), key=lambda n: freq[n], reverse=True)[:5]
bot5 = sorted(range(1, 61), key=lambda n: freq[n])[:5]

# Cores e opacidades por dezena
colors = []
for n, c in zip(nums, counts):
    if n in top5:
        colors.append(ACCENT)
    elif n in bot5:
        colors.append(HIGHLIGHT)
    elif c >= media:
        colors.append(ACCENT)
    else:
        colors.append(MUTED)

fig = go.Figure(go.Bar(
    x=nums,
    y=counts,
    marker_color=colors,
    text=[f"{c}" for c in counts],
    textposition="outside",
    hovertemplate="Dezena <b>%{x}</b><br>Sorteios: <b>%{y}</b><extra></extra>",
    showlegend=False,
))

fig.add_hline(y=media, line_dash="dash", line_color=PRIMARY, line_width=1.5,
              annotation_text=f"Media: {media:.0f}x",
              annotation_position="top right")

fig.update_layout(
    **LAYOUT,
    title="Frequencia historica de cada dezena (1996-2026)",
    xaxis=dict(title="Dezena", tickmode="linear", tick0=1, dtick=1,
               tickfont=dict(size=9), gridcolor="#e0e0e0"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
)
fig.write_html(EXPORTS / "eda_01_frequencia_dezenas.html")
fig.show()

print(f"Mais frequentes : {sorted(top5)}")
print(f"Menos frequentes: {sorted(bot5)}")

Mais frequentes : [5, 10, 27, 37, 53]
Menos frequentes: [15, 21, 22, 26, 55]


> **Leitura:** em ~3.000 sorteios, o desvio padrão esperado por dezena é da ordem de ±√(n·p·(1−p)), onde p ≈ 0,1. Diferenças de 50–100 aparições entre a dezena mais e a menos frequente são compatíveis com aleatoriedade. O gráfico mostra exatamente isso: variação existe, mas sem nenhuma dezena estruturalmente destacada.

### 2. Frequência por faixa de dezena

As dezenas agrupadas em seis faixas de 10 (01–10, 11–20, ..., 51–60). A linha tracejada é o valor esperado se a distribuição for perfeitamente uniforme entre as faixas.

Uma crença comum entre apostadores é que "os números do meio saem mais". Este gráfico testa isso diretamente: se verdade, as faixas 21–30 e 31–40 deveriam se destacar das demais de forma consistente.

In [48]:
dezenas   = all_dezenas()
labels    = ["01-10", "11-20", "21-30", "31-40", "41-50", "51-60"]
bins      = [0, 10, 20, 30, 40, 50, 60]
counts, _ = np.histogram(dezenas, bins=bins)
esperado  = dezenas.size / 6
pcts      = [100 * c / dezenas.size for c in counts]

fig = go.Figure(go.Bar(
    x=labels,
    y=counts,
    marker_color=[ACCENT if c > esperado else MUTED for c in counts],
    text=[f"{c:,}<br>({p:.1f}%)" for c, p in zip(counts, pcts)],
    textposition="outside",
    hovertemplate="Faixa <b>%{x}</b><br>Sorteios: <b>%{y:,}</b><br>%{text}<extra></extra>",
))

fig.add_hline(y=esperado, line_dash="dash", line_color=HIGHLIGHT, line_width=2,
              annotation_text=f"Esperado: {esperado:.0f}",
              annotation_position="top right")

fig.update_layout(
    **LAYOUT,
    title="Sorteios por faixa de dezena",
    xaxis=dict(title="Faixa", gridcolor="#e0e0e0", type="category"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
    showlegend=False,
)
fig.write_html(EXPORTS / "eda_02_frequencia_faixa.html")
fig.show()

### 3. Composição par/ímpar

De 1 a 60, há exatamente 30 pares e 30 ímpares. A quantidade de pares em um sorteio de 6 dezenas segue uma distribuição **hipergeométrica** — matematicamente previsível, sem mistério.

A composição mais provável é **3 pares + 3 ímpares**, mas 2P/4I e 4P/2I também têm probabilidades altas. Composições extremas (6P/0I ou 0P/6I) são raras por construção matemática — não por alguma "regra oculta" do jogo.

> Estratégias que exigem equilíbrio perfeito (3P/3I) jogam com a composição mais comum, mas isso tem um custo: se muita gente escolhe a mesma composição, o rateio em caso de acerto aumenta.

In [49]:
n_pares  = df[DEZENA_COLS].apply(lambda row: (row % 2 == 0).sum(), axis=1)
contagem = n_pares.value_counts().sort_index()
labels   = [f"{p}P / {6-p}I" for p in contagem.index]
pcts     = [100 * v / len(df) for v in contagem.values]

fig = go.Figure(go.Bar(
    x=labels,
    y=contagem.values,
    marker_color=[ACCENT if p == 3 else MUTED for p in contagem.index],
    text=[f"{p:.1f}%" for p in pcts],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Sorteios: <b>%{y:,}</b><br>%{text}<extra></extra>",
))

fig.update_layout(
    **LAYOUT,
    title="Distribuicao de pares e impares por sorteio",
    xaxis=dict(title="Composicao par / impar", gridcolor="#e0e0e0", type="category"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
    showlegend=False,
)
fig.write_html(EXPORTS / "eda_03_pares_impares.html")
fig.show()

n_pares.describe()

count    3002.000000
mean        3.016989
std         1.202316
min         0.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         6.000000
dtype: float64

### 4. Distribuição da soma dos 6 números

A soma dos 6 números sorteados é uma **estatística resumo** que captura muito do perfil de um sorteio de uma vez só. Pelo Teorema Central do Limite, somar variáveis aleatórias independentes converge para uma distribuição normal — mesmo que cada dezena individualmente siga uma uniforme discreta.

O mínimo teórico é 1+2+3+4+5+6 = **21** e o máximo é 55+56+57+58+59+60 = **345**. A média esperada é (1+60)/2 × 6 = **183**.

> **Para o bolão:** apostas com soma muito abaixo de ~130 ou acima de ~230 correspondem às caudas da distribuição — historicamente raras. Isso não as torna impossíveis, mas significa que você está jogando contra a frequência histórica.

In [50]:
soma = df[DEZENA_COLS].sum(axis=1)

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=soma,
    nbinsx=55,
    marker_color=ACCENT,
    opacity=0.85,
    name="Sorteios",
    hovertemplate="Soma: <b>%{x}</b><br>Freq: <b>%{y}</b><extra></extra>",
))

fig.add_vline(x=soma.mean(), line_dash="dash", line_color=HIGHLIGHT, line_width=2,
              annotation_text=f"Media ({soma.mean():.0f})",
              annotation_position="top right")

fig.add_vline(x=soma.median(), line_dash="dot", line_color=PRIMARY, line_width=2,
              annotation_text=f"Mediana ({soma.median():.0f})",
              annotation_position="bottom right")

fig.update_layout(
    **LAYOUT,
    title="Distribuicao da soma dos 6 numeros por sorteio",
    xaxis=dict(title="Soma dos 6 numeros", gridcolor="#e0e0e0"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
    showlegend=False,
)
fig.write_html(EXPORTS / "eda_04_soma_dezenas.html")
fig.show()

soma.describe()

count    3002.000000
mean      183.160227
std        39.974158
min        66.000000
25%       155.000000
50%       184.000000
75%       211.000000
max       331.000000
dtype: float64

---

## Bloco 2 — Comportamento temporal

> **Pergunta central:** o comportamento das dezenas muda ao longo do tempo?

As análises anteriores trataram os 3.000+ sorteios como um bloco estático. Isso é útil para estabilizar frequências, mas pode mascarar variações que ocorreram ao longo dos quase 30 anos de histórico da Mega Sena.

As seções 5 e 6 introduzem a **dimensão temporal**: o intervalo entre aparições de cada dezena (gap) e como a frequência relativa evolui ao longo dos concursos.

### 5. Gap entre aparições de cada dezena

O **gap** é o número de concursos entre duas aparições consecutivas de uma dezena. Se ela aparece em ~10% dos sorteios, o gap médio esperado é de ~10 concursos.

> **Atenção à falácia do jogador:** *"Essa dezena não sai faz 20 concursos — está atrasada!"*
> Essa intuição pressupõe que o sorteio tem memória. Não tem. Cada concurso é **independente**. Um gap alto não aumenta nem diminui a probabilidade futura — ela continua sendo ~10% em cada novo sorteio.

O histograma global mostra como os gaps se distribuem. Em um processo sem memória, essa distribuição se aproxima de uma **exponencial** (ou geométrica discreta): muitos gaps curtos, poucos gaps longos — e gaps muito longos existem e são esperados.

In [51]:
gaps_por_dezena = {}
for num in range(1, 61):
    mask      = df[DEZENA_COLS].isin([num]).any(axis=1)
    concursos = df.loc[mask, "numero"].values
    if len(concursos) > 1:
        gaps_por_dezena[num] = np.diff(concursos)

all_gaps   = np.concatenate(list(gaps_por_dezena.values()))
media_gaps = {n: g.mean() for n, g in gaps_por_dezena.items()}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Distribuicao do gap global",
                                    "Gap medio por dezena"])

# Histograma global
fig.add_trace(go.Histogram(
    x=all_gaps, nbinsx=60,
    marker_color=ACCENT, opacity=0.85, name="Gap",
    hovertemplate="Gap: <b>%{x}</b> concursos<br>Freq: <b>%{y}</b><extra></extra>",
), row=1, col=1)
fig.add_vline(x=all_gaps.mean(), line_dash="dash", line_color=HIGHLIGHT,
              line_width=2, row=1, col=1,
              annotation_text=f"Media: {all_gaps.mean():.1f}")

# Gap medio por dezena
nums_g  = list(media_gaps.keys())
meias_g = list(media_gaps.values())
m_ger   = np.mean(meias_g)
fig.add_trace(go.Bar(
    x=nums_g, y=meias_g,
    marker_color=[ACCENT if v > m_ger else MUTED for v in meias_g],
    name="Gap medio",
    hovertemplate="Dezena <b>%{x}</b><br>Gap medio: <b>%{y:.1f}</b> concursos<extra></extra>",
), row=1, col=2)
fig.add_hline(y=m_ger, line_dash="dash", line_color=HIGHLIGHT,
              line_width=2, row=1, col=2,
              annotation_text=f"Media: {m_ger:.1f}")

fig.update_layout(
    **LAYOUT,
    title="Gap entre aparicoes de cada dezena",
    height=430,
    showlegend=False,
)
fig.update_xaxes(gridcolor="#e0e0e0", type="linear")
fig.update_yaxes(gridcolor="#e0e0e0")
fig.write_html(EXPORTS / "eda_05_gap_dezenas.html")
fig.show()

print(f"Gap global: media={all_gaps.mean():.1f} | mediana={np.median(all_gaps):.1f} | max={all_gaps.max()}")

Gap global: media=10.0 | mediana=7.0 | max=109


### 6. Evolução temporal da frequência

Frequência de cada dezena calculada em uma **janela deslizante de 200 sorteios** (passo de 50). Em vez de olhar o histórico completo de uma vez, acompanhamos como a frequência relativa oscila ao longo do tempo.

Em destaque: as 3 dezenas **historicamente mais frequentes** (vermelho) e as 3 **menos frequentes** (âmbar). As demais ficam em cinza — clique na legenda para isolar traces.

> Se o processo for estacionário e aleatório, as frequências em janela devem oscilar **sem tendência persistente** ao longo do tempo. Qualquer aparente "fase quente" de uma dezena deve ser interpretada com cautela: com 60 dezenas sendo monitoradas simultaneamente, algumas sempre vão parecer estar em ciclo — mesmo em dados completamente aleatórios.

In [52]:
WINDOW = 200
STEP   = 50

freq_total = Counter(all_dezenas())
top3 = [n for n, _ in freq_total.most_common(3)]
bot3 = [n for n, _ in sorted(freq_total.items(), key=lambda x: x[1])[:3]]

janelas = []
for start in range(0, len(df) - WINDOW + 1, STEP):
    bloco      = df.iloc[start: start + WINDOW]
    freq_bloco = Counter(bloco[DEZENA_COLS].values.flatten())
    c_meio     = int(bloco["numero"].median())
    for d in range(1, 61):
        janelas.append({"concurso": c_meio, "dezena": d, "freq": freq_bloco.get(d, 0)})

df_ev = pd.DataFrame(janelas)

fig = go.Figure()

# Fundo: todas as dezenas em cinza
for dezena in range(1, 61):
    if dezena in top3 or dezena in bot3:
        continue
    sub = df_ev[df_ev["dezena"] == dezena].sort_values("concurso")
    fig.add_trace(go.Scatter(
        x=sub["concurso"], y=sub["freq"],
        mode="lines",
        line=dict(color=MUTED, width=0.5),
        opacity=0.25,
        showlegend=False,
        hoverinfo="skip",
    ))

# Destaque: top3 e bot3
for dezena in top3:
    sub = df_ev[df_ev["dezena"] == dezena].sort_values("concurso")
    fig.add_trace(go.Scatter(
        x=sub["concurso"], y=sub["freq"],
        mode="lines",
        name=f"Dezena {dezena} (top)",
        line=dict(color=ACCENT, width=2.5),
        hovertemplate=f"Dezena {dezena}<br>Concurso: %{{x}}<br>Freq: %{{y}}<extra></extra>",
    ))

for dezena in bot3:
    sub = df_ev[df_ev["dezena"] == dezena].sort_values("concurso")
    fig.add_trace(go.Scatter(
        x=sub["concurso"], y=sub["freq"],
        mode="lines",
        name=f"Dezena {dezena} (bottom)",
        line=dict(color=HIGHLIGHT, width=2.5, dash="dash"),
        hovertemplate=f"Dezena {dezena}<br>Concurso: %{{x}}<br>Freq: %{{y}}<extra></extra>",
    ))

fig.update_layout(
    **LAYOUT,
    title=f"Evolucao temporal da frequencia (janela={WINDOW} concursos, passo={STEP})",
    xaxis=dict(title="Num. do concurso", gridcolor="#e0e0e0"),
    yaxis=dict(title=f"Freq. em janela de {WINDOW} sorteios", gridcolor="#e0e0e0"),
    height=480,
)
fig.write_html(EXPORTS / "eda_06_evolucao_temporal.html")
fig.show()

print(f"Top 3: {top3} | Bottom 3: {bot3}")

Top 3: [np.int64(10), np.int64(53), np.int64(37)] | Bottom 3: [np.int64(26), np.int64(21), np.int64(55)]


---

## Bloco 3 — Co-ocorrência

> **Pergunta central:** alguns pares de dezenas saem juntos mais do que o acaso explicaria?

Até aqui analisamos as dezenas individualmente. Agora passamos para **combinações de dois números**: quais duplas aparecem juntas com mais regularidade ao longo da história?

Com C(60,2) = **1.770 pares possíveis** e cada sorteio ativando C(6,2) = **15 pares**, o valor esperado por par é calculável e serve de linha de referência.

### 7. Co-ocorrência de pares de dezenas

Top 30 pares que mais apareceram juntos no mesmo sorteio. A linha tracejada é o **valor esperado** assumindo distribuição uniforme:

```
E[par] = total_sorteios × C(6,2) / C(60,2)  =  total_sorteios × 15 / 1770
```

> **Problema das comparações múltiplas:** com 1.770 pares sendo comparados simultaneamente, é estatisticamente esperado que alguns se destaquem por pura variância aleatória. Um par que aparece 10–15% acima da média não é uma "combinação mágica" — é flutuação normal. Para afirmar associação real, seria necessário corrigir o nível de significância (ex: correção de Bonferroni com α/1770 ≈ 0,00003).

In [53]:
pares = []
for row in df[DEZENA_COLS].itertuples(index=False):
    pares.extend(combinations(sorted(row), 2))

freq_pares = Counter(pares)
top30      = freq_pares.most_common(30)
esperado   = len(df) * 15 / 1770

labels = [f"{a}-{b}" for (a, b), _ in top30]
values = [v for _, v in top30]
desvio = [f"{(v - esperado) / esperado * 100:+.1f}%" for v in values]

fig = go.Figure(go.Bar(
    x=values[::-1],
    y=labels[::-1],
    orientation="h",
    marker_color=[ACCENT if v > esperado else MUTED for v in values[::-1]],
    text=[f"{v}x ({d})" for v, d in zip(values[::-1], desvio[::-1])],
    textposition="outside",
    hovertemplate="Par <b>%{y}</b><br>Co-ocorrencias: <b>%{x}</b><br>%{text}<extra></extra>",
))

# Anotação com referência absoluta (yref="paper" = desvinculado do eixo categórico)
fig.add_annotation(
    x=esperado,
    y=0.98,
    yref="paper",
    text=f"<b>Esperado: {esperado:.0f}x</b>",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor=HIGHLIGHT,
    borderwidth=2,
    borderpad=8,
    font=dict(color=HIGHLIGHT, size=12),
    xanchor="center",
)

layout_custom = dict(**LAYOUT)
layout_custom.update({
    "title": "Top 30 pares de dezenas mais frequentes",
    "xaxis": dict(title="Num. de co-ocorrencias", gridcolor="#e0e0e0"),
    "yaxis": dict(title="Par", gridcolor="#e0e0e0", tickfont=dict(size=10), type="category"),
    "height": 650,
    "showlegend": False,
    "margin": dict(t=70, b=50, l=80, r=120),
})

fig.update_layout(**layout_custom)
fig.write_html(EXPORTS / "eda_07_pares_coocorrencia.html")
fig.show()

print(f"Esperado por par: {esperado:.1f}x")
print(f"Top 5 pares: {top30[:5]}")

Esperado por par: 25.4x
Top 5 pares: [((4, 52), 41), ((5, 27), 41), ((23, 53), 41), ((36, 53), 41), ((38, 53), 40)]


---

## Bloco 4 — Contexto externo e correlações

> **Pergunta central:** fatores externos ao sorteio em si — local onde foi realizado, ordem de saída das bolas, status de acúmulo — têm relação com os resultados?

A **expectativa nula em todos os casos:** se o mecanismo de sorteio é justo e independente, nenhuma dessas variáveis externas deve alterar a distribuição dos números sorteados.

---

### 8. Local dos sorteios

A Caixa realizou sorteios em diferentes cidades e espaços físicos ao longo dos quase 30 anos de histórico. Esta seção investiga se há diferença no comportamento das dezenas entre locais — e como a taxa de acúmulo e o prêmio médio variam.

Volume histórico por local. A Caixa muda o local de sorteio ao longo dos anos — a dominância de um local indica a era principal do jogo.

In [54]:
df_local = df[df["local"].str.strip() != ""].copy()
print(f"{len(df_local)} sorteios com local definido ({100*len(df_local)/len(df):.1f}%)")

top_locais = (
    df_local["local"]
    .value_counts()
    .head(15)
    .sort_values(ascending=True)
)

fig = go.Figure(go.Bar(
    x=top_locais.values,
    y=top_locais.index,
    orientation="h",
    marker_color=[ACCENT if i >= len(top_locais) - 3 else MUTED for i in range(len(top_locais))],
    text=top_locais.values,
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>%{x} sorteios<extra></extra>",
))
fig.update_layout(
    **LAYOUT,
    title="Frequencia de sorteios por local (top 15)",
    xaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    yaxis=dict(title="", tickfont=dict(size=10)),
    height=520,
    showlegend=False,
)
fig.write_html(EXPORTS / "eda_08a_local_frequencia.html")
fig.show()

2849 sorteios com local definido (94.9%)


Taxa de acúmulo e prêmio médio por local. Locais com alta taxa de acúmulo podem refletir períodos de maiores prêmios acumulados ou contexto histórico distinto.

In [55]:
local_stats = (
    df_local
    .groupby("local")
    .agg(
        total=("numero", "count"),
        acumulados=("acumulado", "sum"),
        premio_medio=("premio_total_sena", "mean"),
    )
    .assign(taxa_acum=lambda x: 100 * x["acumulados"] / x["total"])
    .sort_values("total", ascending=False)
    .head(10)
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Taxa de acumulo (%)", "Premio medio da sena (R$ M)"])

fig.add_trace(go.Bar(
    x=local_stats["taxa_acum"],
    y=local_stats.index,
    orientation="h",
    marker_color=[ACCENT if v > local_stats["taxa_acum"].mean() else MUTED
                  for v in local_stats["taxa_acum"]],
    text=[f"{v:.1f}%" for v in local_stats["taxa_acum"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Taxa acumulo: %{x:.1f}%<extra></extra>",
    name="Taxa acumulo",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=local_stats["premio_medio"] / 1e6,
    y=local_stats.index,
    orientation="h",
    marker_color=[ACCENT if v > local_stats["premio_medio"].mean() else MUTED
                  for v in local_stats["premio_medio"]],
    text=[f"R$ {v/1e6:.1f}M" for v in local_stats["premio_medio"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Premio medio: R$ %{x:.1f}M<extra></extra>",
    name="Premio medio",
), row=1, col=2)

fig.update_layout(
    **LAYOUT,
    title="Taxa de acumulo e premio medio por local (top 10 por volume)",
    height=480,
    showlegend=False,
)
fig.update_xaxes(gridcolor="#e0e0e0")
fig.update_yaxes(gridcolor="#e0e0e0")
for ann in fig.layout.annotations:
    ann.font = dict(color=PRIMARY, size=12)
fig.write_html(EXPORTS / "eda_08b_local_acumulo_premio.html")
fig.show()

local_stats[["total", "acumulados", "taxa_acum", "premio_medio"]].round(1)

,total,acumulados,taxa_acum,premio_medio
local,,,,
Caminhão da Sorte,1675,1284,76.7,0.0
ESPAÇO DA SORTE,546,459,84.1,0.0
ESPAÇO LOTERIAS CAIXA,208,165,79.3,0.0
Auditório,191,144,75.4,0.0
CAMINHÃO DA SORTE,73,65,89.0,0.0
Espaço Loterias Caixa,50,42,84.0,0.0
Estúdio de TV,34,21,61.8,0.0
Espaço Caixa Loterias,30,26,86.7,0.0
ESTÚDIO DE TV,18,11,61.1,0.0


Desvio relativo de frequência de cada dezena em relação ao esperado (`6/60 = 10%`) para os 8 locais com mais sorteios. Vermelho = sorteada acima do esperado, cinza = abaixo. Sem viés real esperado — desvios acentuados seriam anomalia estatística.

In [56]:
top8 = local_stats.head(8).index.tolist()
df_top8 = df_local[df_local["local"].isin(top8)]

expected_rate = 6 / 60
heat = np.zeros((60, len(top8)))
for j, loc in enumerate(top8):
    sub  = df_top8[df_top8["local"] == loc]
    n    = len(sub)
    freq = Counter(sub[DEZENA_COLS].values.flatten())
    for d in range(1, 61):
        heat[d - 1, j] = freq.get(d, 0) / n if n > 0 else 0

heat_dev = (heat - expected_rate) / expected_rate

short_labels = [l[:25] + "..." if len(l) > 25 else l for l in top8]

fig = go.Figure(go.Heatmap(
    z=heat_dev,
    x=short_labels,
    y=list(range(1, 61)),
    colorscale=[[0, MUTED], [0.5, BG], [1, ACCENT]],
    zmid=0,
    colorbar=dict(title="Desvio<br>relativo", tickformat=".0%"),
    hovertemplate="Local: <b>%{x}</b><br>Dezena: <b>%{y}</b><br>Desvio: <b>%{z:.1%}</b><extra></extra>",
))
fig.update_layout(
    **LAYOUT,
    title="Desvio relativo de frequencia das dezenas por local (top 8 por volume)",
    xaxis=dict(title="Local", tickfont=dict(size=9), tickangle=-20),
    yaxis=dict(title="Dezena", dtick=5),
    height=720,
)
fig.write_html(EXPORTS / "eda_08c_local_heatmap.html")
fig.show()

### 9. Ordem das bolas sorteadas ao vivo

No sorteio ao vivo, as bolas saem uma a uma e são registradas em sequência (`s1`, `s2`, ..., `s6`). O que vemos no volante são os números em **ordem crescente** — mas a ordem de extração é diferente.

**Hipótese testada:** bolas com números menores tendem a sair nas primeiras posições?

Fisicamente, se todas as bolas têm o mesmo peso e diâmetro e a máquina é calibrada, não há razão para viés. Mas pequenas diferenças de desgaste, temperatura ou pressão do ar poderiam, em tese, criar padrões. Os dados decidem.

P(posição | dezena): dado que a dezena saiu, qual a probabilidade de ter sido em cada posição (s1–s6)? Distribuição uniforme esperada ≈ 16,7% por posição. Desvios persistentes indicariam viés no mecanismo físico de sorteio.

In [57]:
heat_pos = np.zeros((60, 6))
for i, col in enumerate(SORTEIO_COLS):
    for dezena in range(1, 61):
        heat_pos[dezena - 1, i] = (df[col] == dezena).sum()

row_sums = heat_pos.sum(axis=1, keepdims=True)
heat_norm = np.where(row_sums > 0, heat_pos / row_sums, 0)

fig = go.Figure(go.Heatmap(
    z=heat_norm,
    x=["s1", "s2", "s3", "s4", "s5", "s6"],
    y=list(range(1, 61)),
    colorscale=[[0, BG], [0.5, MUTED], [1, ACCENT]],
    colorbar=dict(title="P(pos|dezena)", tickformat=".1%"),
    hovertemplate="<b>Dezena %{y}</b><br>Posicao: <b>%{x}</b><br>P: <b>%{z:.1%}</b><extra></extra>",
))
fig.add_hline(y=20.5, line_dash="dot", line_color=HIGHLIGHT, line_width=0.8, opacity=0.5)
fig.add_hline(y=40.5, line_dash="dot", line_color=HIGHLIGHT, line_width=0.8, opacity=0.5)
fig.update_layout(
    **LAYOUT,
    title="P(posicao de sorteio | dezena) — distribuicao condicional normalizada",
    xaxis=dict(title="Posicao no sorteio ao vivo"),
    yaxis=dict(title="Dezena", dtick=5),
    height=720,
)
fig.write_html(EXPORTS / "eda_09b_ordem_heatmap.html")
fig.show()

Hipótese: dezenas menores tendem a sair em posições iniciais? Rho de Spearman mede a correlação entre posição (1–6) e valor da dezena em cada sorteio. Rho ≈ 0 = ordem aleatória; p-valor do t-test confirma ou rejeita a hipótese.

In [58]:
posicoes = np.arange(1, 7)
rhos = np.array([spearmanr(posicoes, row)[0] for row in df[SORTEIO_COLS].values])
t_stat, p_val = ttest_1samp(rhos, popmean=0)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Distribuicao do rho por sorteio",
                                    "Media movel do rho (janela=100)"])

fig.add_trace(go.Histogram(
    x=rhos, nbinsx=50, name="Rho",
    marker_color=ACCENT, opacity=0.85,
    hovertemplate="Rho: <b>%{x:.2f}</b><br>Freq: <b>%{y}</b><extra></extra>",
), row=1, col=1)
fig.add_vline(x=float(rhos.mean()), line_dash="dash", line_color=HIGHLIGHT, line_width=2,
              row=1, col=1, annotation_text=f"Media: {rhos.mean():.3f}")
fig.add_vline(x=0, line_dash="dot", line_color=PRIMARY, line_width=1,
              row=1, col=1, annotation_text="Zero")

rho_roll = pd.Series(rhos).rolling(100, center=True, min_periods=20).mean()
fig.add_trace(go.Scatter(
    x=df["numero"].values, y=rho_roll.values,
    mode="lines", line=dict(color=ACCENT, width=2),
    hovertemplate="Concurso %{x}<br>Rho medio: %{y:.3f}<extra></extra>",
), row=1, col=2)
fig.add_hline(y=0, line_dash="dot", line_color=PRIMARY, line_width=1,
              row=1, col=2, annotation_text="Zero (aleatorio)")

fig.update_layout(
    **LAYOUT,
    title=(f"Spearman(posicao, valor dezena) | media={rhos.mean():.4f} | "
           f"t={t_stat:.2f} | p={p_val:.4f}"),
    height=440,
    showlegend=False,
)
fig.update_xaxes(gridcolor="#e0e0e0")
fig.update_yaxes(gridcolor="#e0e0e0")
for ann in fig.layout.annotations:
    ann.font = dict(color=PRIMARY, size=12)
fig.write_html(EXPORTS / "eda_09c_ordem_spearman.html")
fig.show()

interp = "dezenas maiores saem depois" if rhos.mean() > 0 else "dezenas menores saem depois"
sig    = "significativo (p<0.05)" if p_val < 0.05 else "nao significativo (p>=0.05)"
print(f"Rho medio: {rhos.mean():.4f} | p-valor: {p_val:.4f}")
print(f"Tendencia: {interp} — {sig}")

Rho medio: -0.0100 | p-valor: 0.2264
Tendencia: dezenas menores saem depois — nao significativo (p>=0.05)


### 10. Correlações e ferramentas exploratórias

Até aqui cada análise focou em uma variável ou relação de cada vez. Nesta seção, olhamos o sorteio como um **vetor de features** e buscamos correlações entre elas — e entre elas e o status de acúmulo do concurso.

Features derivadas de cada sorteio:

| Feature | Descrição |
|---|---|
| `soma` | Soma das 6 dezenas |
| `amplitude` | Diferença entre a maior e a menor (d6 − d1) |
| `n_pares` | Quantidade de números pares |
| `faixa_media` | Média das dezenas (sorteio "alto" ou "baixo") |
| `n_altos` | Dezenas acima de 30 |
| `desvio_pad` | Dispersão interna do sorteio |

**Hipótese nula geral:** o resultado de um sorteio não depende de quantos prêmios foram acumulados anteriormente. As ferramentas abaixo testam isso de diferentes ângulos.

Correlação de Pearson e Spearman entre features derivadas de cada sorteio: `soma` (soma das 6 dezenas), `amplitude` (d6−d1), `n_pares`, `faixa_media`, `n_altos` (dezenas > 30) e `desvio_pad` (dispersão interna do sorteio).

In [59]:
df_feat = df.copy()
df_feat["soma"]        = df[DEZENA_COLS].sum(axis=1)
df_feat["amplitude"]   = df["d6"] - df["d1"]
df_feat["n_pares"]     = df[DEZENA_COLS].apply(lambda r: (r % 2 == 0).sum(), axis=1)
df_feat["faixa_media"] = df[DEZENA_COLS].mean(axis=1)
df_feat["n_altos"]     = df[DEZENA_COLS].apply(lambda r: (r > 30).sum(), axis=1)
df_feat["desvio_pad"]  = df[DEZENA_COLS].std(axis=1)

feat_cols = ["soma", "amplitude", "n_pares", "faixa_media", "n_altos", "desvio_pad"]
corr_p = df_feat[feat_cols].corr(method="pearson")
corr_s = df_feat[feat_cols].corr(method="spearman")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Pearson", "Spearman"])

for ci, mat in enumerate([corr_p, corr_s], start=1):
    fig.add_trace(go.Heatmap(
        z=mat.values,
        x=feat_cols,
        y=feat_cols,
        colorscale=[[0, MUTED], [0.5, BG], [1, ACCENT]],
        zmid=0, zmin=-1, zmax=1,
        text=[[f"{v:.2f}" for v in row] for row in mat.values],
        texttemplate="%{text}",
        textfont=dict(size=10),
        colorbar=dict(x=0.45 if ci == 1 else 1.0, thickness=12),
        showscale=True,
        hovertemplate="<b>%{x}</b> x <b>%{y}</b><br>r = %{z:.3f}<extra></extra>",
    ), row=1, col=ci)

fig.update_layout(
    **LAYOUT,
    title="Matriz de correlacao entre features dos sorteios",
    height=500,
)
for ann in fig.layout.annotations:
    ann.font = dict(color=PRIMARY, size=13)
fig.write_html(EXPORTS / "eda_10a_correlacao_matrix.html")
fig.show()

Distribuição da soma dos 6 números em concursos acumulados vs. não acumulados. KS test verifica se as duas populações têm distribuições distintas — p < 0.05 indicaria que a soma difere entre os grupos.

In [60]:
soma_ac  = df_feat.loc[df_feat["acumulado"],  "soma"]
soma_nac = df_feat.loc[~df_feat["acumulado"], "soma"]
ks_stat, ks_p = ks_2samp(soma_ac, soma_nac)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=soma_ac, nbinsx=50,
    name=f"Acumulado (n={len(soma_ac)})",
    marker_color=ACCENT, opacity=0.65,
    hovertemplate="Soma %{x}<br>Freq: %{y}<extra></extra>",
))
fig.add_trace(go.Histogram(
    x=soma_nac, nbinsx=50,
    name=f"Nao acumulado (n={len(soma_nac)})",
    marker_color=MUTED, opacity=0.65,
    hovertemplate="Soma %{x}<br>Freq: %{y}<extra></extra>",
))
fig.add_vline(x=float(soma_ac.mean()),  line_dash="dash", line_color=ACCENT, line_width=2,
              annotation_text=f"Media acum.: {soma_ac.mean():.0f}",
              annotation_position="top", annotation_yshift=30)
fig.add_vline(x=float(soma_nac.mean()), line_dash="dash", line_color=MUTED,  line_width=2,
              annotation_text=f"Media n.acum.: {soma_nac.mean():.0f}",
              annotation_position="top", annotation_yshift=8)
interp_ks = "distribuicoes diferentes (p<0.05)" if ks_p < 0.05 else "sem diferenca significativa"
fig.update_layout(
    **LAYOUT,
    title=f"Soma: acumulado vs. nao-acumulado | KS={ks_stat:.3f}, p={ks_p:.4f} — {interp_ks}",
    xaxis=dict(title="Soma", gridcolor="#e0e0e0"),
    yaxis=dict(title="Sorteios", gridcolor="#e0e0e0"),
    barmode="overlay",
    height=440,
)
fig.write_html(EXPORTS / "eda_10b_soma_acumulado_ks.html")
fig.show()

print(f"Acumulado     : media={soma_ac.mean():.1f}  std={soma_ac.std():.1f}")
print(f"Nao acumulado : media={soma_nac.mean():.1f}  std={soma_nac.std():.1f}")
print(f"KS: stat={ks_stat:.4f}, p={ks_p:.4f} — {interp_ks}")

Acumulado     : media=185.6  std=39.2
Nao acumulado : media=174.2  std=41.5
KS: stat=0.1180, p=0.0000 — distribuicoes diferentes (p<0.05)


Chi-square testa associação entre composição par/ímpar e acúmulo do concurso. Resíduos padronizados `(obs−exp)/√exp` revelam quais células contribuem mais para a estatística — valores |r| > 2 merecem atenção.

In [61]:
contingency = pd.crosstab(df_feat["n_pares"], df_feat["acumulado"])
chi2_val, chi2_p, dof, expected_cnt = chi2_contingency(contingency)

labels_par = [f"{p}P/{6-p}I" for p in contingency.index]
residuals  = (contingency.values - expected_cnt) / np.sqrt(expected_cnt)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Contagens observadas",
                                    "Residuos padronizados (obs-exp)/sqrt(exp)"])

for ci, (zvals, fmt) in enumerate([(contingency.values, ".0f"), (residuals, ".2f")], start=1):
    fig.add_trace(go.Heatmap(
        z=zvals,
        x=["Nao acumulado", "Acumulado"],
        y=labels_par,
        colorscale=[[0, MUTED], [0.5, BG], [1, ACCENT]],
        zmid=0,
        text=[[f"{v:{fmt}}" for v in row] for row in zvals],
        texttemplate="%{text}",
        textfont=dict(size=11),
        colorbar=dict(x=0.45 if ci == 1 else 1.0, thickness=12),
        showscale=True,
        hovertemplate="Paridade: <b>%{y}</b><br>%{x}<br>%{z}<extra></extra>",
    ), row=1, col=ci)

sig = "associacao significativa (p<0.05)" if chi2_p < 0.05 else "sem associacao significativa"
fig.update_layout(
    **LAYOUT,
    title=f"Chi-square: paridade x acumulado | chi2={chi2_val:.2f}, p={chi2_p:.4f}, dof={dof} — {sig}",
    height=440,
)
for ann in fig.layout.annotations:
    ann.font = dict(color=PRIMARY, size=12)
fig.write_html(EXPORTS / "eda_10c_chi2_paridade.html")
fig.show()

print(f"Chi2={chi2_val:.3f}, p={chi2_p:.4f}, dof={dof}")
print(sig)

Chi2=9.463, p=0.1492, dof=6
sem associacao significativa
